**Download necessary files**

In [1]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /Users/user/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/user/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

**Import the necessary libraries**

In [2]:
import numpy as np
import random
import string

**Import libraries for scraping - beautifulSoup**

In [3]:
from bs4 import BeautifulSoup
import urllib.request
import re

**Read the web link**

In [4]:
# Define URL and headers (pretend to be a real browser)
url = "https://en.wikipedia.org/wiki/Natural_language_processing"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

# Create a request object with headers
req = urllib.request.Request(url, headers=headers)

# Open and read the page
with urllib.request.urlopen(req) as response:
    raw_html = response.read()

raw_html[:500]

b'<!DOCTYPE html>\n<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled wp25easte'

**Convert the web link contents into a file**

In [5]:
# Parse the HTML using BeautifulSoup
article_html = BeautifulSoup(raw_html, "lxml")

**Find only the html text containing inside < p > tags**

In [6]:
# Extract all paragraph tags
article_paragraphs = article_html.find_all("p")

**Read all the text inside paragraphs**

In [7]:
# Combine all text into one string
article_text = ""
for para in article_paragraphs:
    article_text += para.get_text()

article_text[:500]

'\nNatural language processing (NLP) is the processing of natural language information by a computer. NLP is a subfield of computer science and is closely associated with artificial intelligence. NLP is also related to information retrieval, knowledge representation, computational linguistics, and linguistics more broadly.[1]\nMajor processing tasks in an NLP system include: speech recognition, text classification, natural language understanding, and natural language generation.\nNatural language pr'

**Split the sentences**

In [8]:
corpus = nltk.sent_tokenize(article_text)
corpus

['\nNatural language processing (NLP) is the processing of natural language information by a computer.',
 'NLP is a subfield of computer science and is closely associated with artificial intelligence.',
 'NLP is also related to information retrieval, knowledge representation, computational linguistics, and linguistics more broadly.',
 '[1]\nMajor processing tasks in an NLP system include: speech recognition, text classification, natural language understanding, and natural language generation.',
 'Natural language processing has its roots in the 1950s.',
 '[2] Already in 1950, Alan Turing published an article titled "Computing Machinery and Intelligence" which proposed what is now called the Turing test as a criterion of intelligence, though at the time that was not articulated as a problem separate from artificial intelligence.',
 'The proposed test includes a task that involves the automated interpretation and generation of natural language.',
 "The premise of symbolic NLP is often il

**Clean the text**

In [9]:
for i in range(len(corpus )):
    corpus [i] = corpus [i].lower()
    corpus [i] = re.sub(r'\W',' ',corpus [i])
    corpus [i] = re.sub(r'\s+',' ',corpus [i])

**Create a dictionary of word frequency**

In [10]:
wordfreq = {}
for sentence in corpus:
    tokens = nltk.word_tokenize(sentence)
    for token in tokens:
        if token not in wordfreq.keys():
            wordfreq[token] = 1
        else:
            wordfreq[token] += 1

**Filter the top 200 words**

In [11]:
import heapq
most_freq = heapq.nlargest(200, wordfreq, key=wordfreq.get)

### TF-IDF Formula
IDF (Inverse Document Frequency)
```
IDF(term) = log(N / (1 + n_term))
```

*  N = total number of documents
*  n_term = number of documents that contain the term
*  1 + avoids division by zero

TF-IDF
```
TFIDF(term, doc) = TF(term, doc) * IDF(term)
```

*  TF(term, doc) = count of the term in that document (or normalized count)
*  IDF(term) = from the IDF formula above

**Compute IDF values**

In [12]:
word_idf_values = {}
for token in most_freq:
    doc_containing_word = 0
    for document in corpus:
        if token in nltk.word_tokenize(document):
            doc_containing_word += 1
    word_idf_values[token] = np.log(len(corpus)/(1 + doc_containing_word))

**Compute TF values**

In [13]:
word_tf_values = {}
for token in most_freq:
    sent_tf_vector = []
    for document in corpus:
        doc_freq = 0
        for word in nltk.word_tokenize(document):
            if token == word:
                  doc_freq += 1
        word_tf = doc_freq/len(nltk.word_tokenize(document))
        sent_tf_vector.append(word_tf)
    word_tf_values[token] = sent_tf_vector

**Compute TF-IDF values**

In [14]:
tfidf_values = []
for token in word_tf_values.keys():
    tfidf_sentences = []
    for tf_sentence in word_tf_values[token]:
        tf_idf_score = tf_sentence * word_idf_values[token]
        tfidf_sentences.append(tf_idf_score)
    tfidf_values.append(tfidf_sentences)

In [15]:
tf_idf_model = np.asarray(tfidf_values)

In [16]:
tf_idf_model = np.transpose(tf_idf_model)

In [17]:
tf_idf_model

array([[0.02123225, 0.01101076, 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.01101076, 0.04749831, ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.04433175, ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.01025005, 0.00797331, 0.01146511, ..., 0.        , 0.        ,
        0.        ],
       [0.01292398, 0.01340441, 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.00503816, 0.00261272, 0.04508314, ..., 0.        , 0.        ,
        0.        ]], shape=(35, 200))

In [18]:
import pandas as pd
pd.DataFrame(tf_idf_model, columns=most_freq)

,the,of,and,nlp,language,a,in,to,natural,cognitive,...,decline,dominance,chomskyan,linguistic,theories,transformational,whose,theoretical,underpinnings,discouraged
0,0.021232,0.011011,0.000000,0.060521,0.130899,0.065449,0.000000,0.000000,0.152920,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,0.00000,0.00000
1,0.000000,0.011011,0.047498,0.060521,0.000000,0.065449,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,0.00000,0.00000
2,0.000000,0.000000,0.044332,0.056487,0.000000,0.000000,0.000000,0.056487,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,0.00000,0.00000
3,0.000000,0.000000,0.033249,0.042365,0.091629,0.000000,0.049520,0.000000,0.107044,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,0.00000,0.00000
4,0.033028,0.000000,0.000000,0.000000,0.101810,0.000000,0.110044,0.000000,0.118938,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,0.00000,0.00000
5,0.013826,0.003585,0.015465,0.000000,0.000000,0.042618,0.023033,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,0.00000,0.00000
6,0.037156,0.009634,0.041561,0.000000,0.057268,0.057268,0.000000,0.000000,0.066903,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,0.00000,0.00000
7,0.017835,0.006166,0.013300,0.033892,0.018326,0.036652,0.000000,0.016946,0.021409,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,0.00000,0.00000
8,0.016514,0.008564,0.000000,0.000000,0.050905,0.000000,0.000000,0.000000,0.059469,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,0.00000,0.00000
9,0.024771,0.006423,0.000000,0.000000,0.076358,0.038179,0.082533,0.000000,0.044602,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,0.00000,0.00000




---



## Implementation using Sklearn TfidfVectorizer

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [23]:
# Step 1: Initialize TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    max_features=200,            # limit vocab to top 200 most frequent words
    # stop_words='english',        # remove common English stopwords
    # token_pattern=r'\b[a-zA-Z]+\b',  # keep alphabetic words only (ignore numbers)
    # lowercase=True
)

In [24]:
# Step 2: Fit and transform the corpus
X = vectorizer.fit_transform(corpus)

In [25]:
# Step 3: Convert to DataFrame for readability
tfidf_df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
tfidf_df.head()

,19,1980s,1990s,ai,algorithms,among,an,and,applications,approach,...,were,what,which,while,whose,winter,with,word,words,world
0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.183970,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.272082,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.171311,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.243009,0.147380,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
